In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
from torchvision.transforms import v2

In [2]:
train_data = datasets.PCAM(
    root='data',
    split='train',
    download=False,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

In [3]:
train_data_sub = Subset(train_data, range(6400))

In [4]:
train_data_sub[0][0].shape, train_data_sub[0][1]

(torch.Size([3, 96, 96]), 0)

In [5]:
batch_size = 64
t_data = DataLoader(train_data_sub, batch_size=batch_size, shuffle=True)

In [6]:
for X, y in t_data:
    print(X.shape, y.shape, y.dtype)
    break

torch.Size([64, 3, 96, 96]) torch.Size([64]) torch.int64


In [9]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'
device

'mps'

In [14]:
class MyModel(nn.Module):
    def __init__(self, pos_weight):
        super().__init__()
        # Register as a buffer so it moves to GPU/CPU automatically with the module
        self.register_buffer('pos_weight', torch.tensor([pos_weight], dtype=torch.float32))
        # self.layer_stack = nn.Sequential(
        #     nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5, stride=1, padding=0), #92
        #     nn.BatchNorm2d(16),
        #     nn.ReLU(),
        #     nn.MaxPool2d(kernel_size=2), # 46
        #     nn.Conv2d(in_channels=16, out_channels=32, kernel_size=5, stride=1, padding=0), # 42
        #     nn.BatchNorm2d(num_features=32),
        #     nn.ReLU(),
        #     nn.MaxPool2d(kernel_size=2), # 21
        #     nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=0),# 19
        #     nn.BatchNorm2d(num_features=64),
        #     nn.ReLU(),
        #     nn.MaxPool2d(kernel_size=2), #9
        #     nn.Flatten(),
        #     nn.Linear(in_features=64*9*9, out_features=1024),
        #     nn.ReLU(),
        #     nn.Dropout(0.2),
        #     nn.Linear(in_features=1024, out_features=64),
        #     nn.ReLU(),
        #     nn.Dropout(0.2),
        #     nn.Linear(64,2)
        # )

    def forward(self, logits, targets):
        return nn.functional.binary_cross_entropy_with_logits(
            input=logits,
            target=targets.float(),
            pos_weight= self.pos_weight,
        )


In [19]:
if __name__ == "__main__":
    # If positive samples are 5x more important/rare, set pos_weight to 5.0
    criterion = MyModel(pos_weight=5.0).to(device)

    # Simulated model outputs (logits) and true binary targets
    mock_logits = torch.tensor([2.0, -1.0, 3.5, -2.5], requires_grad=True)
    mock_targets = torch.tensor([1.0, 0.0, 1.0, 1.0]) # Note the rare positive misclassifications
    mock_logits  = mock_logits.detach().to(device).requires_grad_()   # keep it a leaf
    mock_targets = mock_targets.to(device)

    # Compute loss
    loss = criterion(mock_logits, mock_targets)
    print(f"Loss value: {loss.item():.4f}")

    # Backward pass works out of the box
    loss.backward()
    print(f"Gradients: {mock_logits.grad}")

Loss value: 3.4978
Gradients: tensor([-0.1490,  0.0672, -0.0366, -1.1552], device='mps:0')
